In [2]:
import pandas as pd

games_df = pd.read_csv("../database data en code/data/Final Database/games.csv")
detailed_games_df = pd.read_csv("../database data en code/data/Final Database/detailed_games.csv")
company_games_df = pd.read_csv("../database data en code/data/Final Database/company_games.csv")
genre_games_df = pd.read_csv("../database data en code/data/Final Database/genre_games.csv")
platform_df = pd.read_csv("../database data en code/data/Final Database/platform.csv")

print(games_df.head())

       id                   slug                    title    released  \
0   58172             spacewar-2                spacewar!  1962-01-01   
1  379437        zeusophthekroos           zeusofthecrows  1970-01-01   
2   56681  the-oregon-trail-1971  the oregon trail (1971)  1971-12-03   
3   55012                 empire                   empire  1971-01-01   
4   58174         computer-space           computer space  1971-12-01   

   metacritic  rating  ratings_count  playtime  added  \
0         NaN    3.79             22         1     40   
1         NaN    0.00              4         0     16   
2         NaN    3.62             34         0     87   
3         NaN    3.43              6         0     31   
4         NaN    2.91              8         0     29   

                                           platforms                 genres  \
0                                                NaN                 Arcade   
1                                                NaN        

In [2]:
detailed_games_df.groupby("year", as_index=False)["rating"].mean()

,year,rating
0,1962,3.790000
1,1970,0.000000
2,1971,2.706667
3,1972,3.440000
4,1973,3.170000
5,1974,3.515000
6,1975,1.126667
7,1976,2.557143
8,1977,2.677692
9,1978,1.140588


In [6]:
from dash import Dash, dcc, html, Input, Output, ctx, ALL, MATCH
import plotly.express as px
import pandas as pd

# --------------------------------------------
# ----------------- Mock Data ----------------
# --------------------------------------------

games = ["GTA V", "Zelda", "Minecraft", "Fortnite", "Skyrim"]
player_counts = [200000, 180000, 160000, 120000, 100000]
years = list(range(1970, 2025))
metrics = ["rating", "twitch_count", "youtube_count", "added", "metacritic"]
genres = ['Action', "Adventure", "Arcade","Board Games","Card","Casual","Educational","Family","Fighting","Indie","Massively Multiplayer","Platformer",
          "Puzzle","Racing","RPG","Shooter","Simulation","Sports","Strategy",]

# "Games", "Genres" nog doen?

df = pd.DataFrame({
    "Game": games * 10,
    "Player Count": [val + (i * 5000 % 15000) for i, val in enumerate(player_counts * 10)]
})

# --------------------------------------------
# -------------- Dash App Setup --------------
# --------------------------------------------

app = Dash(__name__)
app.title = "Dashboard"
app.config.suppress_callback_exceptions = True

# --------------------------------------------
# ---------------- Tabs Config ---------------
# --------------------------------------------

tabs = [
    {'label': 'Most Popular', 'value': 'most-popular'},
    {'label': 'Genres', 'value': 'top games per genre'},
    {'label': 'Companies', 'value': 'companies'},
    {'label': 'New Releases / Coming Soon', 'value': 'new-releases'},
    {'label': '...', 'value': 'placeholder'}
]

# --------------------------------------------
# ---------- Most Popular Tab Layout ---------
# --------------------------------------------

def most_popular_layout():
    return html.Div([

        # Top Filter Bar
        html.Div([
            html.Div("BY:", style={'padding': '0 10px', 'fontWeight': 'bold'}),
            *[html.Button(label, id={'type': 'filter-button', 'index': label},
                          n_clicks=0, style={'margin': '0 5px'}) for label in metrics]
        ], style={
            'display': 'flex', 'padding': '10px',
            'backgroundColor': '#a3d2ff', 'borderBottom': '1px solid #ccc'
        }),

        # Main Layout: Boxplot + Right Sidebar
        html.Div([
            html.Div([
                dcc.Graph(id='barchart')
            ], style={'flex': 1, 'padding': '10px'}),

            html.Div([
                dcc.Input(
                    id="game-search",
                    type="text",
                    placeholder="Search games...",
                    style={'width': '90%', 'margin': '10px'}
                ),
                html.Div(id="game-sidebar", style={
                    'overflowY': 'scroll', 'height': '500px', 'padding': '0 10px'
                })
            ], style={
                'width': '200px', 'backgroundColor': '#a3d2ff', 'paddingTop': '10px'
            })
        ], style={'display': 'flex', 'flexDirection': 'row', 'height': '600px'}),

        html.Div([
            html.Div([
                html.Button(str(year), id={'type': 'year-button', 'index': str(year)}, n_clicks=0, style={'margin': '5px'}) for year in years
            ], style={
                'display': 'flex', 'overflowX': 'scroll', 'padding': '10px 0'
            })
        ], style={
            'backgroundColor': '#a3d2ff', 'borderTop': '1px solid #ccc'
        })
    ])

# --------------------------------------------
# ----------- Companies Tab Layout -----------
# --------------------------------------------

def companies_layout():
    
    return html.Div([
        html.Div([
            dcc.Input(
                id="company-search", type="text", placeholder="Search company...",
                style={'marginLeft': 'auto', 'marginRight': '10px', 'padding': '5px'}
            )
        ], style={
            'display': 'flex', 'alignItems': 'center', 'justifyContent': 'space-between',
            'backgroundColor': '#a3d2ff', 'padding': '10px 20px', 'marginBottom': '20px'
        }),

       html.Div([
            dcc.Graph(id="company-bar-chart")
        ], style={'padding': '0 20px'})
    ])

# --------------------------------------------
# --------- top games per genre Layout--------
# --------------------------------------------

def top_games_per_genre_layout():
    return html.Div([

        # Top Filter Bar
        html.Div([
            html.Div("BY:", style={'padding': '0 10px', 'fontWeight': 'bold'}),
            *[html.Button(label, id={'type': 'genre-button', 'index': label},
                          n_clicks=0, style={'margin': '0 5px'}) for label in genres]
        ], style={
            'display': 'flex', 'padding': '10px',
            'backgroundColor': '#a3d2ff', 'borderBottom': '1px solid #ccc'
        }),

        # Main Layout: Boxplot + Right Sidebar
        html.Div([
            html.Div([
                dcc.Graph(id='barchart-genre')
            ], style={'flex': 1, 'padding': '10px'}),

            html.Div([
                dcc.Input(
                    id="genre-search",
                    type="text",
                    placeholder="Search games...",
                    style={'width': '90%', 'margin': '10px'}
                ),
                html.Div(id="genre-sidebar", style={
                    'overflowY': 'scroll', 'height': '500px', 'padding': '0 10px'
                })
            ], style={
                'width': '200px', 'backgroundColor': '#a3d2ff', 'paddingTop': '10px'
            })
        ], style={'display': 'flex', 'flexDirection': 'row', 'height': '600px'}),
    ])

# --------------------------------------------
# ---------------- Main Layout ---------------
# --------------------------------------------

app.layout = html.Div([
    dcc.Store(id="selected-year"),
    dcc.Store(id="selected-metric", data=metrics[0]),
    dcc.Store(id="selected-genre", data=genres[0]),
    html.Div([
        html.H1("Dashboard", style={'margin': '10px 20px'}),
        dcc.Tabs(
            id="main-tabs", value='most-popular',
            children=[dcc.Tab(label=tab['label'], value=tab['value']) for tab in tabs],
            style={'margin': '0 20px'}
        )
    ], style={'backgroundColor': '#f0f0f0', 'paddingBottom': '10px'}),

    html.Div(id='tab-content', style={'padding': '20px'})
])

# --------------------------------------------
# ------ Callback to Switch Tab Content ------
# --------------------------------------------

@app.callback(Output('tab-content', 'children'), Input('main-tabs', 'value'))
def render_content(tab):
    if tab == 'most-popular':
        return most_popular_layout()
    elif tab == "top games per genre":
        return top_games_per_genre_layout()
    elif tab == 'companies':
        return companies_layout()
    elif tab == 'new-releases':
        return html.Div([html.H3("New Releases / Coming Soon")])
    elif tab == 'placeholder':
        return html.Div([html.H3("Future Tabs")])
    return html.Div()

# Callback to store selected metric and year
@app.callback(
    Output("selected-metric", "data"),
    Input({'type': 'filter-button', 'index': ALL}, 'n_clicks'),
    prevent_initial_call=True
)
def store_metric(clicks):
    triggered = ctx.triggered_id
    if triggered and triggered['type'] == 'filter-button':
        return triggered['index']
    return metrics[0]

@app.callback(
    Output("selected-year", "data"),
    Input({'type': 'year-button', 'index': ALL}, 'n_clicks'),
    prevent_initial_call=True
)
def store_year(clicks):
    triggered = ctx.triggered_id
    if triggered and triggered['type'] == 'year-button':
        return int(triggered['index'])
    return None

@app.callback(
    Output("barchart", "figure"),
    Input("selected-metric", "data"),
    Input("selected-year", "data")
)
def update_chart(metric, selected_year):
    filtered_df = detailed_games_df.copy()
    if selected_year:
        filtered_df = filtered_df[filtered_df["year"] == selected_year]
    filtered_df = filtered_df.sort_values(by=metric, ascending=False)
    fig = px.bar(
        filtered_df, x="title", y=metric,
        title=f"{metric} for {selected_year}" if selected_year else f"{metric} per Game"
    )
    return fig

@app.callback(
    Output("game-sidebar", "children"),
    Input("selected-year", "data"),
    Input("selected-metric", "data"),
    Input("game-search", "value")
)
def update_sidebar(year, metric, query):
    df = detailed_games_df.copy()
    if year:
        df = df[df["year"] == year]
    if query:
        df = df[df["title"].str.contains(query, case=False)]
    df_sorted = df.sort_values(by=metric, ascending=False)
    return [
        html.Div(f"{i+1}. {row['title']}", style={
            'padding': '5px', 'borderBottom': '1px solid #ccc'})
        for i, row in df_sorted.iterrows()
    ]


# genre callbacks

@app.callback(
    Output("selected-genre", "data"),
    Input({'type': 'genre-button', 'index': ALL}, 'n_clicks'),
    prevent_initial_call=True
)
def store_genre(clicks):
    triggered = ctx.triggered_id
    if triggered and triggered['type'] == 'genre-button':
        return triggered['index']
    return genres[0]

@app.callback(
    Output("barchart-genre", "figure"),
    Input("selected-genre", "data"),
)

def update_chart_genre(genre):
    filtered_df = genre_games_df.copy()
    filtered_df = filtered_df.drop_duplicates()
    filtered_df = filtered_df[filtered_df["main_genre"] == genre]
    filtered_df = filtered_df.sort_values(by="rating", ascending=False)
    fig = px.bar(
        filtered_df, x="title", y="rating",
        title=f"top rated games of {genre} genre"
    )
    return fig

@app.callback(
    Output("genre-sidebar", "children"),
    Input("selected-genre", "data"),
    Input("genre-search", "value")
)

def update_sidebar_genre(genre, query):
    df = genre_games_df.copy()
    df = df.drop_duplicates()
    df = df[df["main_genre"]== genre]
    df_sorted = df.sort_values(by="rating", ascending=False)
    if query:
        df_sorted = df_sorted[df_sorted["title"].str.contains(query, case=False)]
    return [
        html.Div(f"{i+1}. {row['title']}", style={
            'padding': '5px', 'borderBottom': '1px solid #ccc'})
        for i, row in df_sorted.iterrows()
    ]


# company callbacks

@app.callback(
    Output("company-bar-chart", "figure"),
    Input("company-search", "value")
)
def update_company_bar_chart(company_name):
    if not company_name:
        return px.bar(title="Enter a company to view its games")

    df = company_games_df.copy()
    
    # Case-insensitive match
    filtered_df = df[df["company"].str.contains(company_name, case=False, na=False)]

    if filtered_df.empty:
        return px.bar(title=f"No games found for '{company_name}'")

    sorted_df = filtered_df.sort_values(by="rating", ascending=False)

    fig = px.bar(
        sorted_df,
        x="title",
        y="rating",
        title=f"Games by '{company_name}' sorted by rating",
        labels={"title": "Game Title", "rating": "Rating"}
    )
    fig.update_layout(xaxis_tickangle=-45)
    return fig


# --------------------------------------------
# -------------------- Run -------------------
# --------------------------------------------
if __name__ == '__main__':
    app.run(debug=True)


In [75]:
import pandas as pd

# =============================
# Dataset Loading
# =============================

games_df = pd.read_csv("../database data en code/data/Final Database/games.csv")
detailed_games_df = pd.read_csv("../database data en code/data/Final Database/detailed_games.csv")
company_games_df = pd.read_csv("../database data en code/data/Final Database/company_games.csv")
genre_games_df = pd.read_csv("../database data en code/data/Final Database/genre_games.csv")
platform_df = pd.read_csv("../database data en code/data/Final Database/platform.csv")

C:\Users\laure\AppData\Local\Temp\ipykernel_14640\1589539965.py:8: DtypeWarning:

Columns (8,26,86) have mixed types. Specify dtype option on import or set low_memory=False.



In [7]:
import dash
from dash import dcc, html, Input, Output, State, ctx, ALL
import dash_bootstrap_components as dbc
import threading
import webbrowser
import plotly.express as px

# Create Dash app with Bootstrap theme
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app.title = "Dashboard"

years = list(range(1970, 2025))

# =========================
# Helper component builders
# =========================

def build_top_control(active_tab):
    return dbc.ButtonGroup([
        dbc.Button("Games", id="games-button", n_clicks=0, color="primary" if active_tab == "games" else "secondary"),
        dbc.Button("Companies", id="companies-button", n_clicks=0, color="primary" if active_tab == "companies" else "secondary")
    ], id="top_control", style={"width": "100%"})

def build_search_bar():
    return dbc.Card(
        dbc.CardBody([
            dbc.Input(
                id="search_bar",
                placeholder="Search...",
                type="text",
                style={"marginBottom": "0"}
            )
        ]),
        style={
            "marginTop": "1rem",
            "marginBottom": "8rem",
            "boxShadow": "0px 2px 6px rgba(0,0,0,0.1)",
            "border": "1px solid #ced4da",
            "borderRadius": "0.5rem",
            "backgroundColor": "white"
        }
    )


def build_data_view():
    return html.Div(
        children=[
            html.H1("Main Visualization Area"),
            html.Div(
                dcc.Graph(id="main_graph", style={"height": "600px", "width": "100%"}),
                style={
                    "overflowX": "auto",
                    "width": "100%",
                    "paddingBottom": "1rem"
                }
            )
        ],
        id="data_view",
        style={"padding": "2rem"}
    )



def build_games_middle(selected_sub, selected_sort_options, selected_genres):
    # Left Column (always the same)
    buttons = []
    options = ["Most Popular", "Genres", "New Releases"]
    for label in options:
        idx = label.lower().replace(" ", "-") + "-sub"
        color = "primary" if selected_sub == idx else "secondary"
        buttons.append(
            dbc.Button(
                label,
                id={"type": "sub-button", "index": idx},
                color=color,
                outline=False,
                n_clicks=0,
                style={"width": "100%", "marginBottom": "0.5rem"}
            )
        )

    left_col = html.Div(buttons)

    # Right Column (dynamic depending on selected sub-button)
    if selected_sub == "most-popular-sub":
        sort_options = ["Rating", "YouTube", "Twitch", "Added", "Metacritic"]
        right_col = html.Div([
            html.H6("Sort By:"),
            *[
                dbc.Button(
                    m,
                    id={"type": "sort-button", "index": m},
                    color="primary" if m in selected_sort_options else "secondary",
                    outline=False,
                    style={"width": "100%", "marginBottom": "0.25rem"}
                )
                for m in sort_options
            ],
            html.Hr(),
            html.H6("Select Year Range:"),
            dcc.RangeSlider(
                id="year-range-slider",
                min=1970, max=2024, step=1,
                marks={str(y): str(y) for y in range(1970, 2025, 5)},
                value=[2000, 2020],
                vertical=True,
                verticalHeight=300
            )

        ], style={"paddingLeft": "1rem"})

    elif selected_sub == "genres-sub":
        genre_options = ["Action", "Shooter", "Farming"]
        right_col = html.Div([
            html.H6("Genres:"),
            *[
                dbc.Button(
                    g,
                    id={"type": "genre-button", "index": g},
                    color="primary" if g in selected_genres else "secondary",
                    outline=False,
                    style={"width": "100%", "marginBottom": "0.5rem"}
                )
                for g in genre_options
            ]
        ], style={"paddingLeft": "0.5rem"})

    else:
        right_col = html.Div()  # Empty for New Releases etc.

    return dbc.Row([dbc.Col(left_col, width=6), dbc.Col(right_col, width=6)])

def build_companies_middle(selected_sub):
    buttons = []
    options = ["Companies", "Publishers"]
    for label in options:
        idx = label.lower().replace(" ", "-") + "-sub"
        color = "primary" if selected_sub == idx else "secondary"
        buttons.append(
            dbc.Button(
                label,
                id={"type": "sub-button", "index": idx},
                color=color,
                outline=False,
                n_clicks=0,
                style={"width": "100%", "marginBottom": "0.5rem"}
            )
        )
    return html.Div(buttons)

def build_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres):
    return html.Div([
        build_top_control(active_tab),
        html.Hr(),
        html.Div(
            id="middle_options",
            children=build_games_middle(selected_sub, selected_sort_options, selected_genres) if active_tab == "games" else build_companies_middle(selected_sub),
            style={"flexGrow": 1, "overflowY": "auto","overflowX": "hidden","paddingTop": "1rem","paddingBottom": "1rem"}
        ),
        html.Hr(),
        build_search_bar()
    ], style={
        "display": "flex",
        "flexDirection": "column",
        "height": "100%",
        "padding": "1rem"
    })


# =========================
# App Layout
# =========================

# Important: Default sidebar must be built immediately at startup!
initial_active_tab = "games"
initial_selected_sub = "most-popular-sub"

app.layout = dbc.Container(
    fluid=True,
    children=[
        dcc.Store(id="active_main_tab", data=initial_active_tab),
        dcc.Store(id="selected_sub_button", data=initial_selected_sub),
        dcc.Store(id="selected_sort_options", data=[]),
        dcc.Store(id="selected_genres", data=[]),
        dcc.Store(id="selected_year_range", data=[1970, 2024]),

        dbc.Row([
            # Sidebar
            dbc.Col(
                id="sidebar",
                children=build_sidebar(initial_active_tab, initial_selected_sub, [], []),
                width=3,
                style={
                    "backgroundColor": "#f8f9fa",
                    "height": "100vh",
                    "padding": 0,
                    "borderRight": "1px solid #dee2e6",
                    "display": "flex",
                    "flexDirection": "column"
                }
            ),

            # Data View
            dbc.Col(build_data_view(), width=9)
        ])
    ]
)

# =========================
# Callbacks
# =========================

@app.callback(
    [Output("active_main_tab", "data"),
     Output("selected_sub_button", "data")],
    [Input("games-button", "n_clicks"),
     Input("companies-button", "n_clicks"),
     Input({"type": "sub-button", "index": ALL}, "n_clicks")],
    [State("active_main_tab", "data"),
     State("selected_sub_button", "data")]
)
def handle_clicks(games_clicks, companies_clicks, sub_clicks, current_tab, selected_sub):
    triggered = ctx.triggered_id

    if triggered == "games-button":
        return "games", "most-popular-sub"
    elif triggered == "companies-button":
        return "companies", "companies-sub"
    elif isinstance(triggered, dict) and triggered.get("type") == "sub-button":
        return current_tab, triggered["index"]
    else:
        return current_tab, selected_sub

@app.callback(
    Output("sidebar", "children"),
    [Input("active_main_tab", "data"),
     Input("selected_sub_button", "data"),
     Input("selected_sort_options", "data"),
     Input("selected_genres", "data")]
)
def update_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres):
    return build_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres)

@app.callback(
    Output("selected_genres", "data"),
    Input({"type": "genre-button", "index": ALL}, "n_clicks"),
    State("selected_genres", "data"),
    prevent_initial_call=True
)
def toggle_genre_selection(n_clicks_list, selected_genres):
    triggered = ctx.triggered_id
    if not triggered:
        return dash.no_update

    genre = triggered["index"]
    if genre in selected_genres:
        selected_genres.remove(genre)
    else:
        selected_genres.append(genre)

    return selected_genres

@app.callback(
    Output("selected_sort_options", "data"),
    Input({"type": "sort-button", "index": ALL}, "n_clicks"),
    State("selected_sort_options", "data"),
    prevent_initial_call=True
)
def select_sort_option(n_clicks_list, selected_sort_options):
    triggered = ctx.triggered_id
    if not triggered:
        return dash.no_update

    sort_option = triggered["index"]
    return [sort_option]

@app.callback(
    Output("selected_year_range", "data"),
    Input("year-range-slider", "value"),
    prevent_initial_call=True
)
def update_selected_year_range(year_range):
    return year_range


@app.callback(
    Output("main_graph", "figure"),
    [Input("active_main_tab", "data"),
     Input("selected_sub_button", "data"),
     Input("selected_sort_options", "data"),
     Input("selected_year_range", "data")]
)
def update_main_graph(active_tab, selected_sub_button, selected_sort_options, selected_year_range):
    if active_tab == "companies" and selected_sub_button == "companies-sub":
        company_counts = company_games_df["company"].value_counts().reset_index()
        company_counts.columns = ["Company", "Number of Games"]

        company_counts = company_counts.sort_values("Number of Games", ascending=False)

        fig = px.bar(
            company_counts,
            x="Company",
            y="Number of Games",
            title="Top Companies by Number of Games",
            labels={"Company": "Company", "Number of Games": "Number of Games"},
        )

        fig.update_layout(
            xaxis_tickangle=-45,
            height=600,
            margin=dict(l=50, r=30, t=50, b=150),
            bargap=0.2,
        )

        return fig

    elif active_tab == "games" and selected_sub_button == "most-popular-sub":
        # Filter detailed_games_df
        df = detailed_games_df.copy()
        df = df.drop_duplicates()
        df = df[(df["year"] >= selected_year_range[0]) & (df["year"] <= selected_year_range[1])]

        # If no sort option selected yet, default to rating
        if not selected_sort_options:
            sort_by = "rating"
        else:
            sort_by = selected_sort_options[0].lower()  # because button text is like "YouTube"

        # Make sure to map to correct column names if needed
        column_mapping = {
            "rating": "rating",
            "youtube": "youtube_count",
            "twitch": "twitch_count",
            "added": "added",
            "metacritic": "metacritic"
        }
        sort_column = column_mapping.get(sort_by, "rating")

        df = df.sort_values(by=sort_column, ascending=False)
        df = df.head(50)

        fig = px.bar(
            df,
            x="title",
            y=sort_column,
            title=f"Games sorted by {sort_by.capitalize()}",
            labels={"title": "Game", sort_column: sort_by.capitalize()}
        )

        fig.update_layout(
            xaxis_tickangle=-45,
            height=600,
            margin=dict(l=50, r=30, t=50, b=150),
            bargap=0.2,
        )

        return fig

    else:
        return px.bar(title="Select a tab to view data")


# =========================
# Run server
# =========================

def open_browser():
    webbrowser.open_new("http://127.0.0.1:8050/")

if __name__ == "__main__":
    threading.Timer(1, open_browser).start()
    app.run(debug=True, use_reloader=False)

